# A：GTSRB 七类 · 原图单分支 · 三种增强 × 三个种子（Kaggle 版）

这是分工中 A 的正式训练：`raw` 输入 × `repeat_clean / targeted_motion / mixed` × 种子 `42/43/44`，共 9 次，每次 10 轮。
跑完再做**开发集十三条件评估**（13 条件 × 840 张 × 9 个模型）和汇总表格。

**预计耗时**：试跑约 2 分钟，9 次训练约 3–4 小时（T4），评估约 10 分钟。

**需要先做的事**

1. 把 `a_kaggle_dataset.zip` 上传成一个 Kaggle Dataset（Data → Your Datasets → New Dataset），然后在本 Notebook 右侧 **Add Data** 里挂上它。
2. 右侧 **Settings → Accelerator** 选 `GPU T4 ×2`（或 `GPU P100`）。
3. 右侧 **Settings → Internet** 打开（用来 `git clone` 代码；权重和数据都在你上传的 Dataset 里）。

**重要边界**：Kaggle 的框架构建、GPU 和浮点执行都和 Windows 的 RTX 4060 Ti 不同，所以这份结果**不能**和 C 已有的 15 次结果混进同一个三种子均值。报告里要分开写。

**跑完要下载**：最后一步生成的 `/kaggle/working/a_kaggle_results.zip`。

## 1. 参数与路径

只改「开关」和「代码来源」这一段，路径一般不用动。

In [ ]:
import os
import sys
import glob
import json
import hashlib
import shutil
import subprocess
import zipfile
import tarfile
from pathlib import Path

# 这些环境变量必须在任何 cuBLAS 运算之前设好，否则确定性算法会在 CUDA 上报警。
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("NVIDIA_TF32_OVERRIDE", "0")

# ---------------- 开关 ----------------
RUN_SMOKE = True    # 1 轮工程试跑，约 2 分钟
RUN_TRAIN = True    # 九次正式训练，约 3–4 小时
RUN_EVAL = True     # 开发集十三条件评估，约 10 分钟
RUN_REPORT = True   # 表格与强度曲线
RUN_TEST7 = False   # 官方七类测试：等全组方案冻结后再改 True

# ---------------- 代码来源 ----------------
BRANCH = "a/data-protocol-evaluation"
REPO_URL = "https://github.com/yinianqingxian/MLproject.git"

# ---------------- 路径（一般不用改）----------------
WORK = os.environ.get("A_KAGGLE_WORK") or (
    "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
_overridden_inputs = [p for p in (os.environ.get("A_KAGGLE_INPUT") or "").split(os.pathsep) if p]
INPUTS = _overridden_inputs or ["/kaggle/input"]
REPO = Path(WORK) / "MLproject"

STUDY = "results/a_kaggle_v1"
SMOKE = "results/a_kaggle_smoke_v1"
EVAL = "results/a_eval13_kaggle_v1"
REPORT = "report/a_part_kaggle"
ENV_DIR = "results/a_kaggle_environment"
FREEZE = "results/a_test7_freeze.json"
TEST_OUT = "results/a_test7_kaggle_v1"

THREADS = 4
WEIGHTS_SHA = "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec"
GT_SHA = "58a660fab5465cce66d7334eb9422ae09ed16186ad4500766b59b5a077a1c12f"
SPLIT_SHA = "de914a9199138474115a14776a61574651653ed87c72a26b9804bb35c5aa5359"

print("WORK   =", WORK)
print("INPUTS =", INPUTS)
print("REPO   =", REPO)

## 2. 小工具

`run()` 会打印实际执行的命令并在失败时停下（不会静默跳过）。

In [ ]:
def run(command, cwd=None, check=True):
    command = [str(part) for part in command]
    print("$ " + " ".join(command), flush=True)
    process = subprocess.run(command, cwd=str(cwd or REPO), text=True)
    if check and process.returncode != 0:
        raise SystemExit("命令失败（exit %d）：%s" % (process.returncode, " ".join(command)))
    return process


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def count_ppm(directory):
    return sum(1 for _ in Path(directory).glob("*.ppm"))


def flatten_repo():
    # 解压出来的代码可能在 MLproject/ 子目录里，摊平一次。
    if (REPO / "experiments").is_dir():
        return
    nested = REPO / "MLproject"
    if (nested / "experiments").is_dir():
        for item in nested.iterdir():
            shutil.move(str(item), str(REPO / item.name))
        nested.rmdir()

## 3. 环境检查

必须是 GPU 运行时。协议固定 FP32、batch 128、10 轮，**不要**为了跑得快改成 AMP 或减小 batch。

In [ ]:
import platform
import torch
import torchvision

print(platform.platform())
print("python    ", sys.version.split()[0])
print("torch     ", torch.__version__, "| torchvision", torchvision.__version__,
      "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version())

if not torch.cuda.is_available():
    raise SystemExit("没有可用的 GPU：请在右侧 Settings → Accelerator 选择 GPU T4 ×2 或 GPU P100，然后重新运行本 Notebook。")

properties = torch.cuda.get_device_properties(0)
print("GPU       ", properties.name,
      "| %.1f GiB" % (properties.total_memory / 1024 ** 3),
      "| capability %d.%d" % (properties.major, properties.minor))
if torch.cuda.device_count() > 1:
    print("检测到 %d 张 GPU；本协议只用 device 0，不改成多卡也不改 batch。" % torch.cuda.device_count())

In [ ]:
import importlib.util

PACKAGES = {"cv2": "opencv-python-headless", "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
missing = [name for name in PACKAGES if importlib.util.find_spec(name) is None]
if missing:
    print("缺少依赖，安装：", [PACKAGES[name] for name in missing])
    run([sys.executable, "-m", "pip", "install", "-q"] + [PACKAGES[name] for name in missing], cwd=WORK)

import cv2
import sklearn
import matplotlib
print("opencv", cv2.__version__, "| sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

## 4. 取代码

优先 `git clone` 分支（需要 Internet）。没有 Internet 时，退回使用你上传的代码 zip；两者都失败就停在这里并告诉你在找什么。

In [ ]:
def extract_code_archive():
    for base in INPUTS:
        for path in glob.glob(os.path.join(base, "**", "*.zip"), recursive=True):
            try:
                with zipfile.ZipFile(path) as archive:
                    names = archive.namelist()
            except zipfile.BadZipFile:
                continue
            if not any(name.endswith("experiments/run_resnet_study.py") for name in names):
                continue
            print("从压缩包解压代码：", path)
            REPO.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(path) as archive:
                archive.extractall(REPO)
            flatten_repo()
            return True
    return False


if (REPO / "experiments/run_resnet_study.py").is_file():
    print("代码已就位：", REPO)
elif os.environ.get("A_KAGGLE_REPO"):
    print("从 A_KAGGLE_REPO 复制代码")
    shutil.copytree(os.environ["A_KAGGLE_REPO"], REPO, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(".git", "__pycache__", ".test_tmp"))
else:
    clone = run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], cwd=WORK, check=False)
    if clone.returncode != 0:
        print("git clone 失败（多半是没开 Internet），改用上传的代码压缩包")
        if not extract_code_archive():
            raise SystemExit("既没有 Internet 也没有代码压缩包。请打开 Settings → Internet，"
                             "或把 a_kaggle_code.zip 作为 Dataset 挂上来。")

print("代码目录：", REPO)
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=str(REPO),
                            capture_output=True, text=True, check=True).stdout.strip()
    print("代码版本（git commit）：", commit)
except (OSError, subprocess.CalledProcessError):
    print("代码版本：来自上传的压缩包（没有 git 信息）")
run([sys.executable, "-c", "import sys; sys.path.insert(0, 'experiments'); import a_conditions; print('a_conditions ok, 条件数 =', a_conditions.CONDITION_COUNT)"])

## 5. 定位并校验数据

支持三种形态：Dataset 已自动解压、Dataset 里是一个 zip/tgz、或你已经手动解压过。校验数量与官方标注指纹，对不上就停下。

In [ ]:
def find_data_root(bases):
    for base in bases:
        # 深度 0 覆盖「解压后 base 本身就是数据根」，深度 1–4 覆盖 Dataset 的各种嵌套。
        for depth in range(0, 9):
            pattern = os.path.join(base, *(["*"] * depth), "GTSRB", "Final_Training")
            for match in glob.glob(pattern):
                root = Path(match).parent.parent
                if (root / "GTSRB/Final_Test/Images").is_dir() and (root / "GT-final_test.csv").is_file():
                    return root
    return None


def archive_has_data(members):
    has_labels = any(name.endswith("GT-final_test.csv") for name in members)
    has_images = any("Final_Training/Images/" in name for name in members)
    return has_labels and has_images


def extract_data_archive():
    target = Path("/kaggle/tmp/gtsrb") if os.path.isdir("/kaggle/tmp") else Path(WORK) / "_gtsrb"
    candidates = []
    for base in INPUTS:
        for pattern in ("**/*.zip", "**/*.tgz", "**/*.tar.gz"):
            candidates.extend(sorted(glob.glob(os.path.join(base, pattern), recursive=True)))
    for path in candidates:
        print("检查压缩包：", path)
        if path.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(path) as archive:
                    if not archive_has_data(archive.namelist()):
                        continue
                target.mkdir(parents=True, exist_ok=True)
                print("解压到", target, "（约 1–2 分钟）")
                with zipfile.ZipFile(path) as archive:
                    archive.extractall(target)
            except zipfile.BadZipFile:
                continue
        else:
            try:
                with tarfile.open(path) as archive:
                    if not archive_has_data(archive.getnames()):
                        continue
                target.mkdir(parents=True, exist_ok=True)
                print("解压到", target, "（约 1–2 分钟）")
                with tarfile.open(path) as archive:
                    try:
                        archive.extractall(target, filter="data")
                    except TypeError:
                        archive.extractall(target)
            except tarfile.TarError:
                continue
        found = find_data_root([str(target)])
        if found is not None:
            return found
    return None


root = find_data_root(INPUTS)
if root is not None:
    print("数据集已解压：", root)
else:
    print("输入目录里没有现成的图片树，尝试从压缩包解压")
    root = extract_data_archive()

if root is None:
    available = []
    for base in INPUTS:
        if os.path.isdir(base):
            available.extend(sorted(os.listdir(base)))
    raise SystemExit("没找到 GTSRB 数据。请确认 a_kaggle_dataset.zip 已作为 Dataset 挂到本 Notebook。"
                     "当前输入目录内容：" + str(available))

train_dir = root / "GTSRB/Final_Training/Images"
test_dir = root / "GTSRB/Final_Test/Images"
train_count = sum(count_ppm(directory) for directory in train_dir.iterdir() if directory.is_dir())
test_count = count_ppm(test_dir)
print("训练图", train_count, "张 | 测试图", test_count, "张")
if (train_count, test_count) != (39209, 12630):
    raise SystemExit("图片数量不对：期望 39209 / 12630，实际 %d / %d。数据没有完整解压或上传不完整。"
                     % (train_count, test_count))

gt_sha = sha256(root / "GT-final_test.csv")
print("GT-final_test.csv sha256 =", gt_sha)
if gt_sha != GT_SHA:
    raise SystemExit("官方测试标注指纹不对，说明这份数据不是项目用的同一份 GTSRB。")
print("数据校验通过")

In [ ]:
split_path = REPO / "results/pilot_split.csv"
split_sha = sha256(split_path)
print("pilot_split.csv sha256 =", split_sha)
if split_sha != SPLIT_SHA:
    raise SystemExit("冻结划分指纹不对。不要用改名、重新生成或裁剪过的划分文件。")
print("冻结划分校验通过（训练 2940 张 / 98 条轨迹，验证 840 张 / 28 条轨迹）")

## 6. 官方 ResNet18 初始化权重

优先用 Dataset 里带的那一份（我已经放进 `a_kaggle_dataset.zip`），找不到才联网下载。无论哪种都会校验 SHA256。

In [ ]:
weights = REPO / "assets/weights/resnet18-f37072fd.pth"
weights.parent.mkdir(parents=True, exist_ok=True)

if weights.is_file() and sha256(weights) == WEIGHTS_SHA:
    print("权重已就位")
else:
    found = None
    for base in INPUTS:
        for candidate in glob.glob(os.path.join(base, "**", "resnet18-f37072fd.pth"), recursive=True):
            found = candidate
            break
        if found:
            break
    if found is None:
        local_copy = Path(root) / "resnet18-f37072fd.pth"
        if local_copy.is_file():
            found = str(local_copy)
    if found:
        print("从", found, "复制权重")
        shutil.copy2(found, weights)
    else:
        print("联网下载官方权重")
        run(["curl", "-fL", "https://download.pytorch.org/models/resnet18-f37072fd.pth",
             "-o", str(weights)], cwd=WORK)

actual = sha256(weights)
print("weights sha256 =", actual)
if actual != WEIGHTS_SHA:
    raise SystemExit("权重指纹不对：期望 %s，实际 %s" % (WEIGHTS_SHA, actual))
print("权重校验通过")

## 7. 数据协议审计

这一步会重放冻结划分、增强索引和干扰随机序列，断言轨迹无交叉、预算正确，并记录本环境的 `data_fingerprint`。**它通过才说明这批数据能直接用于正式训练。**

In [ ]:
run([sys.executable, "tools/audit_a_data.py",
     "--data-root", root, "--output", ENV_DIR + "/data_audit"])

audit = json.loads((REPO / ENV_DIR / "data_audit/audit.json").read_text(encoding="utf-8"))
print(json.dumps({
    "split_sizes": audit["checks"]["split_sizes"],
    "track_counts": audit["checks"]["track_counts"],
    "overlapping_tracks": audit["checks"]["overlapping_tracks"],
    "training_entries": audit["checks"]["training_entries"],
    "validation_entries": audit["checks"]["validation_entries"],
    "evaluation_conditions": audit["evaluation_budget"]["conditions"],
    "evaluation_pairs": audit["evaluation_budget"]["image_condition_pairs"],
}, indent=2, ensure_ascii=False))
print("本环境 data_fingerprint =", audit["data_fingerprint"])
REFERENCE_FINGERPRINT = "e494bcb968ea8cbaa48c1790f73dcc0a403a8a4be3f16f8740151618f8029eaf"
if audit["data_fingerprint"] == REFERENCE_FINGERPRINT:
    print("与 Windows 参考环境的像素指纹一致：这份数据和既有结果可以逐像素对齐。")
else:
    print("与 Windows 参考环境（e494bcb9…）不同：不同 OpenCV/NumPy 构建下的正常差异。")
    print("同一环境内部仍然自洽，但跨环境的配对比较必须在报告里单独说明，不能直接当成方法差异。")

## 8. 工程试跑（1 轮，不计入正式结果）

只证明数据、权重、显存和保存流程能走通。看到 `Verified 1 completed official ResNet18 runs` 就说明通了。

In [ ]:
if RUN_SMOKE:
    run([sys.executable, "experiments/run_resnet_study.py",
         "--inputs", "raw", "--augmentations", "mixed", "--seeds", "42", "--epochs", "1",
         "--threads", str(THREADS), "--device", "cuda",
         "--data-root", root, "--output-dir", SMOKE])
    run([sys.executable, "experiments/run_resnet_study.py",
         "--inputs", "raw", "--augmentations", "mixed", "--seeds", "42", "--epochs", "1",
         "--threads", str(THREADS), "--device", "cuda",
         "--data-root", root, "--output-dir", SMOKE, "--verify-only"])
    print("试跑通过")
else:
    print("RUN_SMOKE = False，跳过")

## 9. 正式九次训练

`raw` × `repeat_clean / targeted_motion / mixed` × 种子 `42/43/44`，每次 10 轮、350 次更新。
如果这个目录已经存在（上次跑到一半），会自动改用 `--resume` 只补没完成的运行。

**这一格要跑 3–4 小时。** 建议用 `Save Version → Save & Run All (Commit)` 让它后台跑完，不要关浏览器等。

In [ ]:
TRAIN_ARGS = [
    "--inputs", "raw",
    "--augmentations", "repeat_clean", "targeted_motion", "mixed",
    "--seeds", "42", "43", "44",
    "--epochs", "10",
    "--threads", str(THREADS),
    "--device", "cuda",
    "--data-root", root,
    "--output-dir", STUDY,
]

if RUN_TRAIN:
    command = [sys.executable, "experiments/run_resnet_study.py"] + TRAIN_ARGS
    if (REPO / STUDY).exists():
        print("检测到已有研究目录，使用 --resume 只补未完成的运行")
        command.append("--resume")
    run(command)
    print("九次训练结束")
else:
    print("RUN_TRAIN = False，跳过")

## 10. 只读核验

检查 9 个运行的配置、产物和文件指纹。正常输出 `Verified 9 completed official ResNet18 runs`。

In [ ]:
if RUN_TRAIN or (REPO / STUDY).exists():
    run([sys.executable, "experiments/run_resnet_study.py"] + TRAIN_ARGS + ["--verify-only"])
else:
    print("还没有训练结果，跳过")

## 11. 开发集十三条件评估

加载九个已训练的最终权重，在 840 张开发验证图上跑 13 个条件（干净 + 4 族 × 3 强度），不需要重新训练。

In [ ]:
if RUN_EVAL:
    run([sys.executable, "experiments/evaluate_a13.py",
         "--study", STUDY, "--output", EVAL, "--device", "cuda",
         "--data-root", root, "--expect-runs", "9"])
    print("十三条件评估结束")
else:
    print("RUN_EVAL = False，跳过")

## 12. 汇总表格与强度曲线

输出到 `report/a_part_kaggle/`：三种子均值、逐种子配对差值、干扰族均值、逐类 F1、成本表和四族强度曲线。

In [ ]:
if RUN_REPORT:
    run([sys.executable, "experiments/summarize_a_study.py", "--eval", EVAL, "--output", REPORT])
    print("汇总完成：", REPO / REPORT)
else:
    print("RUN_REPORT = False，跳过")

## 13. 数量验收并打包

先按协议核对数量，全部对上才打包。**少一行都会停下来**，避免把不完整的产物当成结果带走。

In [ ]:
def csv_rows(path):
    with open(path, encoding="utf-8") as handle:
        return max(0, sum(1 for _ in handle) - 1)


def sum_rows(runs_root, name):
    return sum(csv_rows(directory / name) for directory in runs_root if (directory / name).is_file())


study_dir = REPO / STUDY
study_runs = sorted(p for p in (study_dir / "runs").iterdir() if p.is_dir())
train_metrics = sum_rows(study_runs, "metrics.csv")
train_predictions = sum_rows(study_runs, "predictions.csv")
train_features = sum(len(list((p / "features").glob("*.npz"))) for p in study_runs)

eval_dir = REPO / EVAL
eval_runs = sorted(p for p in eval_dir.iterdir() if p.is_dir())
eval_metrics = sum_rows(eval_runs, "metrics.csv")
eval_predictions = sum_rows(eval_runs, "predictions.csv")
eval_features = sum(len(list(p.glob("*.npz"))) for p in eval_runs)

table = {
    "训练运行目录": (len(study_runs), 9),
    "训练指标行 (5×9)": (train_metrics, 45),
    "训练逐图预测 (4200×9)": (train_predictions, 37800),
    "训练特征文件 (6×9)": (train_features, 54),
    "评估指标行 (13×9)": (eval_metrics, 117),
    "评估逐图预测 (10920×9)": (eval_predictions, 98280),
    "评估特征文件 (13×9)": (eval_features, 117),
}
bad = []
for label, (actual, expected) in table.items():
    flag = "OK " if actual == expected else "BAD"
    if actual != expected:
        bad.append("%s: 实际 %d，期望 %d" % (label, actual, expected))
    print("%s %-28s %8d / %d" % (flag, label, actual, expected))
if bad:
    raise SystemExit("数量验收未通过：\n" + "\n".join(bad))
print("\n数量全部对上")

In [ ]:
archive_path = Path(WORK) / "a_kaggle_results.zip"
manifest = {
    "study": STUDY, "eval": EVAL, "report": REPORT,
    "environment": json.loads((REPO / ENV_DIR / "data_audit/audit.json").read_text(encoding="utf-8"))["environment"],
    "data_fingerprint": json.loads((REPO / ENV_DIR / "data_audit/audit.json").read_text(encoding="utf-8"))["data_fingerprint"],
    "counts": {label: actual for label, (actual, _) in table.items()},
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "note": "Kaggle environment; do not mix these three-seed means with the Windows or Spark results.",
}

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for relative in (STUDY, EVAL, ENV_DIR, SMOKE, REPORT, TEST_OUT):
        base = REPO / relative
        if not base.exists():
            continue
        for path in sorted(base.rglob("*")):
            if path.is_file():
                archive.write(path, ("MLproject/" + path.relative_to(REPO).as_posix()))
    archive.writestr("MANIFEST.json", json.dumps(manifest, indent=2, ensure_ascii=False))

print("打包完成：", archive_path)
print("大小：%.1f MB" % (archive_path.stat().st_size / 1024 ** 2))
print("在右侧 Output 面板里下载 a_kaggle_results.zip 即可。")

## 14. 官方七类测试（现在先跳过）

按协议要等全组方案冻结后才打开测试分数。到那时把最上面的 `RUN_TEST7` 改成 `True` 并重跑本 Notebook（`--resume` 会复用已有训练，不会重新训练），或者只跑这两格。

In [ ]:
if RUN_TEST7:
    if not (REPO / FREEZE).exists():
        run([sys.executable, "experiments/evaluate_official_test.py",
             "--init-freeze", FREEZE, "--study", STUDY, "--gt", root / "GT-final_test.csv"])
    run([sys.executable, "experiments/evaluate_official_test.py",
         "--freeze", FREEZE, "--output", TEST_OUT, "--device", "cuda",
         "--data-root", root, "--gt", root / "GT-final_test.csv"])
    print("官方七类测试结束：", REPO / TEST_OUT)
else:
    print("RUN_TEST7 = False，跳过官方测试。冻结后再来开这一格。")

## 跑完之后

1. 下载 `/kaggle/working/a_kaggle_results.zip`。
2. 把 `report/a_part_kaggle/dev13_summary.md` 和 `tables/`、`figures/` 发给 D，他做配对 bootstrap 与聚类。
3. 报告里这一部分要写清楚：**环境是 Kaggle（T4），与 Windows / Spark 的结果分开记录，不混合均值。**

### 常见问题

| 现象 | 处理 |
|---|---|
| 没有可用 GPU | Settings → Accelerator → GPU T4 ×2 |
| 找不到数据 | 确认 Dataset 已 Add Data；或确认上传的是 `a_kaggle_dataset.zip` |
| `git clone` 失败 | Settings → Internet 打开；或用 `a_kaggle_code.zip` |
| 训练中断 | 重新运行第 9 格，会自动 `--resume` 跳过已完成运行；半成品会被拒绝，那就把 `STUDY` 改成 `results/a_kaggle_v2` 重跑 |
| `Study code fingerprints differ` | 别在训练途中改 `experiments/` 下 C 的 5 个核心文件 |
| `RuntimeError: … deterministic` | 把完整报错发我，不要擅自改成 `warn_only` 继续跑正式结果 |
| 12 小时上限 | 9 次训练约 3–4 小时，正常不会碰到；真碰到就 `--resume` |